In [ ]:
from torch.utils.data import DataLoader, random_split
import torchvision

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import matplotlib.pyplot as plt
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from torchsummary import summary

from torch.utils.tensorboard import SummaryWriter

import torch.optim 

from tqdm import tqdm

import copy
  
from datetime import datetime

import PIL

import random

SEED = 692


np.random.seed(SEED)
torch.manual_seed(SEED)

# Se estiver usando GPU:
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Dados

## Caminhos

In [ ]:
datasets_path     = '/homeLocal/praticas-cv-cnn/datasets/'
models_path       = '/homeLocal/praticas-cv-cnn/models/'
tensorboard_path  = '/homeLocal/praticas-cv-cnn/Tensorboard/2026_1/lenet5/'

## Dataloader

In [ ]:
def my_imshow(img, dataset, numImages=10):
  
    if dataset == 'cifar10' : 
        img = img / 2 + 0.5     # unnormalize
    
    img = torchvision.utils.make_grid(img[:numImages],nrow=numImages//2)
    
    npimg = img.numpy()    
    npimg = np.transpose(npimg, (1, 2, 0))
    
    plt.axis('off')
    plt.imshow(npimg)
    plt.show()

def show_images(data_loader, dataset, numImages=10) :
    print(f"Train samples, {data_loader['train']['length']}")
    # get some random training images
    dataiter = iter(data_loader['train']['data'])
    images = next(dataiter)[0]
    my_imshow(images, dataset, numImages)

    print(f"Val samples, {data_loader['val']['length']}")
    # get some random val images
    dataiter = iter(data_loader['val']['data'])
    images = next(dataiter)[0]
    my_imshow(images, dataset, numImages)
    
    print(f"Test samples, {data_loader['test']['length']}")
    # get some random training images
    dataiter = iter(data_loader['test']['data'])
    images = next(dataiter)[0]
    my_imshow(images, dataset, numImages)

def get_data_cifar10 ( batch_size , show_image=False, numImages=10 ) :
  
    my_transform = torchvision.transforms.Compose([
                            torchvision.transforms.Resize(28),
                            torchvision.transforms.ToTensor(),
                            torchvision.transforms.Normalize(mean=[0.5],std=[0.5])
                                    ])

    test_dataset = torchvision.datasets.CIFAR10(
                                root=f'{datasets_path}/test/',
                                train=False, 
                                transform=my_transform, 
                                download=False
                                )
    full_train_dataset = torchvision.datasets.CIFAR10(
                                root=f'{datasets_path}/train/', 
                                train=True, 
                                transform=my_transform, 
                                download=False
                                )

    train_ratio = 0.8
    train_size = int(train_ratio * len(full_train_dataset))
    val_size = len(full_train_dataset) - train_size

    train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_dataset, 
                                batch_size=batch_size,
                                shuffle=True
                                )
    val_loader = DataLoader(val_dataset, 
                            batch_size=batch_size,
                            shuffle=False
                            )
    test_loader = DataLoader(test_dataset, 
                            batch_size=batch_size,
                            shuffle=False
                            )

    data_loader = {
        'train': {'data':train_loader, 'length':len(train_dataset)},
        'test' : {'data':test_loader , 'length':len(test_dataset) },
        'val'  : {'data':val_loader  , 'length':len(val_dataset)  },
        'name' : 'cifar10',
        'n_channels' : 3,
        'num_classes' : 10
    }
    
    if show_image :
        show_images(data_loader, data_loader['name'], numImages)
    
    return data_loader


def get_data_mnist ( batch_size , show_image=False, numImages=10 ) :
  
    full_train_dataset = torchvision.datasets.mnist.MNIST(
                            root=f'{datasets_path}/train/', 
                            train=True, 
                            transform=torchvision.transforms.ToTensor(), 
                            download=False
                            )
    test_dataset = torchvision.datasets.mnist.MNIST(
                            root=f'{datasets_path}/test/',
                            train=False, 
                            transform=torchvision.transforms.ToTensor(), 
                            download=False
                            )

    train_ratio = 0.8
    train_size = int(train_ratio * len(full_train_dataset))
    val_size = len(full_train_dataset) - train_size 

    train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    data_loader = {
        'train': {'data':train_loader, 'length':len(train_dataset)},
        'test' : {'data':test_loader , 'length':len(test_dataset) },
        'val'  : {'data':val_loader  , 'length':len(val_dataset)  },
        'name' : 'mnist',
        'n_channels' : 1,
        'num_classes' : 10
    }
    
    if show_image :
        show_images(data_loader, data_loader['name'], numImages)
    
    return data_loader

In [ ]:
get_data_mnist(batch_size=256, show_image=True, numImages=16);

In [ ]:
get_data_cifar10(batch_size=256, show_image=True, numImages=10);

# Rede

## Arquitetura

In [ ]:
class LeNet(nn.Module) :
    def __init__(self, num_classes=10, in_channels=1):
        super(LeNet, self).__init__()
        self.c1 = nn.Conv2d(in_channels=in_channels, out_channels=6, 
                            kernel_size=(5,5), padding=2, stride=1)
        self.s2 = nn.AvgPool2d(kernel_size=2, stride=2)
        self.activationS2 = nn.Sigmoid()
        self.c3 = nn.Conv2d(in_channels=6, out_channels=16, 
                            kernel_size=(5,5), padding=0, stride=1)
        self.s4 = nn.AvgPool2d(kernel_size=2, stride=2)
        self.activationS4 = nn.Sigmoid()
        self.c5 = nn.Conv2d(in_channels=16, out_channels=120, 
                            kernel_size=(5,5), padding=0, stride=1)
        self.activationC5 = nn.Sigmoid()
        self.f6 = nn.Linear(in_features=120, out_features=84)
        self.activationF6 = nn.Tanh()
        self.f = nn.Linear(in_features=84, out_features=num_classes)
        self.softmax = nn.Softmax(dim=1)
        
    def forward(self, x, debug=False):
        # Convolutional
        if debug : print('Input',x.shape)
        y = self.c1(x)
        if debug : print('Outputs:')
        if debug : print('C1',y.shape)
        y = self.s2(y)
        if debug : print('S2',y.shape)
        y = self.activationS2(y)
        if debug : print('Activation S2',y.shape)
        y = self.c3(y)
        if debug : print('C3',y.shape)
        y = self.s4(y)
        if debug : print('S4',y.shape)
        y = self.activationS4(y)
        if debug : print('Activation S4',y.shape)
        # Flattening
        y = self.c5(y)
        if debug : print('C5',y.shape)
        y = self.activationC5(y)
        if debug : print('Activation C5',y.shape)
        y = y.view(y.shape[0], -1)
        if debug : print('Reshape C5',y.shape)
        # Fully Connected
        y = self.f6(y)
        if debug : print('F6',y.shape)
        y = self.activationF6(y)
        if debug : print('Activation F6',y.shape)
        y = self.f(y)
        if debug : print('F',y.shape)
        s = self.softmax(y)
        if debug : print('Softamx',s.shape)
        return y, s

## Informações sobre a rede

In [ ]:
if torch.cuda.is_available():
    my_device = torch.device("cuda:0")
else:
    my_device = torch.device("cpu")
    
print(f"Running on {my_device.type}.")

net = LeNet(num_classes=10, in_channels=3)
net = net.to(my_device)

a = torch.rand( (1, 3, 28, 28) )

b = net( a.to(my_device), debug=True )

In [ ]:

summary(net, input_size=(3,28,28), batch_size=256)
del net, a, b

## Treinamento

In [ ]:
def train ( data_loader, epochs=100, lr=1e-1, prefix='', upper_bound=99.0, device='cpu',
           save=False, debug=False, plot_histograms=False, lambda_reg=0) :

    net = LeNet( data_loader['num_classes'], data_loader['n_channels'] )
    net.to(device)

    optimizer = torch.optim.SGD(net.parameters(), lr, weight_decay=lambda_reg)
    loss = nn.CrossEntropyLoss()

    now = datetime.now()
    suffix = now.strftime("%Y%m%d_%H%M%S")
    prefix = prefix + '-' + suffix if prefix != '' else suffix

    writer = SummaryWriter( log_dir=tensorboard_path+prefix )
    
    writer.add_graph(net, next(iter(data_loader['train']['data']))[0].to(my_device))

    accuracies = []
    max_accuracy = -1.0

    dataset_size = data_loader['train']['length']

    for epoch in tqdm( range(epochs) , desc='Training epochs...' ) :
        net.train()  
        for idx, (train_x, train_label) in enumerate(data_loader['train']['data']):
            train_x = train_x.to(device)
            train_label = train_label.to(device)

            optimizer.zero_grad()

            predict_y = net( train_x )[0]

            # Loss:
            error = loss( predict_y , train_label.long() )

            writer.add_scalar( 'Loss/train', error, 
                            idx+( epoch*(dataset_size//batch_size) ) )

            error.backward()
            optimizer.step()

            # Accuracy:
            predict_ys = torch.max( predict_y, axis=1 )[1]
            correct    = torch.sum(predict_ys == train_label)

            writer.add_scalar( 'Accuracy/train', correct/train_x.size(0)*100, 
                            idx+( epoch*(dataset_size//batch_size) ) )

            if debug and idx % 10 == 0 :
                print(f'idx: {idx}, _error: {error}')

        if plot_histograms : 
            plot_histograms_tensorboard(writer, net, epoch)
        
        accuracy_val, error_val = validate(net, data_loader['val'], device=device, criterion=loss)
        
        accuracies.append(accuracy_val.cpu())
        writer.add_scalar( 'Accuracy/val', accuracy_val, epoch )
    
        if accuracy_val > max_accuracy:
            best_model = copy.deepcopy(net)
            max_accuracy = accuracy_val
            print(f'Saving Best Model with Accuracy: {max_accuracy:3.4f}')
            
        print( f'Epoch: {epoch+1:3d} | Accuracy : {accuracy_val:3.4f}%' )

        if accuracy_val > upper_bound :
            break
   
    if save : 
        path = f'{models_path}-{prefix}-{max_accuracy:.2f}.pkl'
        torch.save(best_model.state_dict(), path)
        print('Model saved in:',path)
  
    plt.plot(accuracies)

    writer.flush()
    writer.close()
    
    return best_model

In [ ]:
def plot_histograms_tensorboard ( writer, net, epoch ) :
    writer.add_histogram('Bias/conv1',   net.c1.bias,        epoch)
    writer.add_histogram('Weight/conv1', net.c1.weight,      epoch)
    writer.add_histogram('Grad/conv1',   net.c1.weight.grad, epoch)

    writer.add_histogram('Bias/conv3',   net.c3.bias,        epoch)
    writer.add_histogram('Weight/conv3', net.c3.weight,      epoch)
    writer.add_histogram('Grad/conv3',   net.c3.weight.grad, epoch)

    writer.add_histogram('Bias/conv5',   net.c5.bias,        epoch)
    writer.add_histogram('Weight/conv5', net.c5.weight,      epoch)
    writer.add_histogram('Grad/conv5',   net.c5.weight.grad, epoch)

## Validação

In [ ]:
def validate ( model , data , device='cpu', criterion=None, confusion_matrix_labels=None) :
    model.eval()
    correct = 0
    # sum = 0
    error = 0

    label_cm = np.array([])
    predicted_cm = np.array([])
    
    for idx, (test_x, test_label) in enumerate(data['data']) : 
        test_x = test_x.to(device)
        test_label = test_label.to(device)
        predict_y = model( test_x )[0].detach()
        predict_ys = torch.max( predict_y, axis=1 )[1]
        # sum = sum + test_x.size(0)
        correct = correct + torch.sum(predict_ys == test_label)

        if criterion != None:
            error = error + criterion( predict_y , test_label )

        if confusion_matrix_labels != None :
            label_cm = np.concatenate((label_cm, test_label.cpu().numpy()))
            predicted_cm = np.concatenate((predicted_cm, predict_ys.cpu().numpy()))

    accuracy = correct*100./data['length']

    if confusion_matrix_labels != None :

        
        cm = confusion_matrix(
                label_cm, 
                predicted_cm, 
                normalize='true'
            )
        cm = np.round(cm*100, 1)
        disp = ConfusionMatrixDisplay(cm, display_labels=confusion_matrix_labels)
        disp.plot(cmap='Blues')
        plt.title('Normalized Confusion Matrix')
        plt.show()

        cm = confusion_matrix(
                label_cm, 
                predicted_cm, 
            )
        disp = ConfusionMatrixDisplay(cm, display_labels=confusion_matrix_labels)
        disp.plot(cmap='Blues')
        plt.title('Confusion Matrix')
        plt.show()
    
    if criterion == None:
        return accuracy
    else :
        return accuracy, error

# Execução

## Treina

In [ ]:
if torch.cuda.is_available():
    my_device = torch.device("cuda:0")
else:
    my_device = torch.device("cpu")

epochs = 10
dataset = 'cifar10' 
# dataset = 'mnist' 
lr = 1.3e0
lambda_reg = 0

if dataset == 'mnist' :
    batch_size = 256
    data_loader = get_data_mnist(batch_size, show_image=True)
elif dataset == 'cifar10' :
    batch_size = 256
    data_loader = get_data_cifar10(batch_size, show_image=True)
else :
    print('Dataset loader not implemented.')
    

prefix = 'LeNet-{}-e-{}-lr-{}'.format(dataset, epochs, lr)

In [ ]:
print(f"Running on {my_device.type}.")

net = train( data_loader=data_loader, epochs=epochs, lr=lr, prefix=prefix , upper_bound=100, device=my_device,
            save=True, debug=False, plot_histograms=True, lambda_reg=lambda_reg )

# Carregar Rede de arquivo

In [ ]:
# del net

path = '/homeLocal/praticas-cv-cnn/models/-LeNet-cifar10-e-10-lr-1.3-20260430_155550-39.34.pkl'
# path = '/homeLocal/praticas-cv-cnn/models/-LeNet-mnist-e-10-lr-1.3-20260429_173134-93.87.pkl'
n_channels = 3

if torch.cuda.is_available():
    my_device = torch.device("cuda:0")
else:
    my_device = torch.device("cpu")


def load_LeNet ( device , path ) :
    net = LeNet(num_classes=10, in_channels=n_channels)
    net = net.to(device)
    net.load_state_dict(torch.load(path))
    net.eval()
    return net

net = load_LeNet(my_device, path)

# Carregar dado do MNIST e inferir

In [ ]:
def sample_and_predict ( data_loader ) :

    test_dataset = data_loader['test']['data'].dataset
    idx = np.random.randint(0, len(test_dataset) - 1)
    img, label = test_dataset[idx]
    
    x = img.unsqueeze_(0)

    x = x.to(my_device)

    y = net ( x )[1]
    confidence = torch.max(y, 1)[0]
    prediction = torch.max(y, 1)[1]

    print( 'Sample: {}'.format(idx) )
    
    img = img.squeeze_(0)

    if dataset=='cifar10':

        img_show = img * 0.5 + 0.5
        plt.imshow(img_show.permute(1, 2, 0))
    else :
        img_show = img.squeeze()  
        img_show = img_show * 0.5 + 0.5
        
        plt.imshow(img_show, cmap='gray')
        
    plt.axis('off')
    plt.show()

    confidence = confidence.data.cpu().numpy()[0]
    prediction = prediction.data.cpu().numpy()[0]

    return prediction, confidence, label

prediction, confidence, label = sample_and_predict(data_loader)
print( f'\nPredicted clas: {prediction} \nClassifier confidence: {confidence*100:4.2f}% \nTrue label: {label}' )


In [ ]:
accuracy_test = validate ( net , 
                          data_loader['test'] , 
                          device=my_device, 
                          confusion_matrix_labels=['0','1','2','3','4','5','6','7','8','9']
                         )
print(f"A acurácia do modelo treinado no conjunto de teste é: {accuracy_test:.2f}% ")